### 1. Install Unsloth and Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-deps --force-reinstall unsloth unsloth_zoo
!pip install --no-deps xformers trl peft accelerate bitsandbytes

### 2. Load Base Model and Tokenizer
Swap `model_name` to train other models. Pick models with instruct safetensors.

In [ ]:
save_filename = "finetunes/qwen3_8b"
model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
datasets = {
    "gad7": "gad7_valid_dataset.json",
    "dass": "dass_valid_dataset.json",
    "ace": "ace_valid_dataset.json",
    "hamd": "hamd_valid_dataset.json",
    "pss14": "pss14_valid_dataset.json",
    "qids": "qids_valid_dataset.json"
}

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192 # Lower to save memory
dtype = None # None for auto detection.
load_in_4bit = True

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    # Change model here
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 3. Prepare Dataset

In [ ]:
# Load the dataset
import json
import os

all_data = []
for task, filename in datasets.items():
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Required dataset {filename} not found for {task}.")

    with open(filename) as f:
        task_data = json.loads('\n'.join([line for line in f]))
        for entry in task_data:
            entry['task'] = task
        all_data.extend(task_data)

# Split the data into conversation data, expected values, and tasks
conversations = [entry['conversation'] for entry in all_data]
expected_outputs = [entry['expected_output'] for entry in all_data]
tasks = [entry['task'] for entry in all_data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')

In [ ]:
from sklearn.model_selection import train_test_split

# Further split the data into training and test data with stratified sampling
conv_train, _conv_test, score_train, _score_test, task_train, _task_test = train_test_split(
    conversations, expected_outputs, tasks, random_state=1234, stratify=tasks
)
print(f"Train samples: {len(conv_train)}, Test samples: {len(_conv_test)}")

In [ ]:
from datasets import Dataset
import json

prompts = {
    "gad7": """
You are a GAD-7 survey scoring assistant.
Your job is to extract answers for the GAD-7 based on a given conversation transcript.
The Generalized Anxiety Disorder 7-item scale (GAD-7) is a clinical assessment tool that measures the severity of anxiety symptoms over the past two weeks.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1   Feeling nervous, anxious, or on edge
q2   Not being able to stop or control worrying
q3   Worrying too much about different things
q4   Trouble relaxing
q5   Being so restless that it's hard to sit still
q6   Becoming easily annoyed or irritable
q7   Feeling afraid, as if something awful might happen

After q7, the assistant asks a follow-up about how much these feelings have interfered with daily life (work, home, relationships). This is the functional impairment question and is NOT scored 0-3 — it is recorded as one of these four exact strings:
  "Not difficult at all"
  "Somewhat difficult"
  "Very difficult"
  "Extremely difficult"

Scale mapping for q1-q7:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
- For q1 through q7, return a single string value: "0", "1", "2", or "3".
- For functional_impairment, return one of the four exact strings listed above.
Rules:
- Match each topic to the correct q number based on the descriptions above. The interviewer may ask multiple follow-up questions for the same item; treat the user's overall answer for that topic as the value.
- Map natural-language frequencies to the closest scale value (e.g., "not at all" / "never" -> 0, "a few days" / "once or twice" / "several days" -> 1, "more than half the days" / "most days" -> 2, "every day" / "nearly every day" / "constantly" / "all the time" -> 3).
- Map the final functional impairment answer to the closest of the four labels (e.g., a user saying "it's been quite a bit, very difficult" -> "Very difficult"; "barely affects me" -> "Not difficult at all"; "I'm drowning" / "completely overwhelmed" -> "Extremely difficult").
- Score each item based solely on what the user said.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "functional_impairment": "Not difficult at all"
}
""",
    "dass": """
You are a DASS-21 survey scoring assistant.
Your job is to extract answers for the DASS-21 based on a given conversation transcript.
The Depression Anxiety Stress Scales 21-item version (DASS-21) is a clinical assessment tool that measures the severity of depression, anxiety, and stress over the past week.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1  (Stress):      Hard to wind down or switch off
q6  (Stress):      Over-reacting to situations
q8  (Stress):      Using a lot of nervous energy
q11 (Stress):      Getting agitated
q12 (Stress):      Difficult to relax
q14 (Stress):      Impatient or intolerant
q18 (Stress):      Touchy or irritable
q2  (Anxiety):     Dryness of mouth
q4  (Anxiety):     Difficulty breathing
q7  (Anxiety):     Trembling
q9  (Anxiety):     Worried about panicking or embarrassing self
q15 (Anxiety):     Close to panic
q19 (Anxiety):     Aware of heart beating without physical exertion
q20 (Anxiety):     Scared without a clear reason
q3  (Depression):  Couldn't experience any positive feeling
q5  (Depression):  Difficult to work up initiative
q10 (Depression):  Nothing to look forward to
q13 (Depression):  Down-hearted or blue
q16 (Depression):  Unable to become enthusiastic
q17 (Depression):  Not worth much as a person
q21 (Depression):  Life felt meaningless

Scale mapping:
0 = never
1 = sometimes
2 = often
3 = almost always

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- Read each user response in order. The first user response answers q1, the second answers q6, the third answers q8, and so on following the order above.
- Score each response based solely on what the user said.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q21 key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q6": "0",
  "q8": "0",
  "q11": "0",
  "q12": "0",
  "q14": "0",
  "q18": "0",
  "q2": "0",
  "q4": "0",
  "q7": "0",
  "q9": "0",
  "q15": "0",
  "q19": "0",
  "q20": "0",
  "q3": "0",
  "q5": "0",
  "q10": "0",
  "q13": "0",
  "q16": "0",
  "q17": "0",
  "q21": "0"
}
""",
    "ace": """
You are an ACE (Adverse Childhood Experiences) survey scoring assistant.
Your job is to extract answers for the ACE questionnaire based on a given conversation transcript.
The ACE questionnaire assesses 10 categories of adverse childhood experiences before age 18.

ACE question order:
q1:  Did a parent or other adult in the household often or very often swear at you, insult you, put you down, or humiliate you? Or act in a way that made you afraid you might be physically hurt?
q2:  Did a parent or other adult in the household often or very often push, grab, slap, or throw something at you? Or ever hit you so hard that you had marks or were injured?
q3:  Did an adult or person at least 5 years older than you ever touch or fondle you or have you touch their body in a sexual way? Or attempt or actually have oral, anal, or vaginal intercourse with you?
q4:  Did you often or very often feel that no one in your family loved you or thought you were important or special? Or your family didn't look out for each other, feel close to each other, or support each other?
q5:  Did you often or very often feel that you didn't have enough to eat, had to wear dirty clothes, and had no one to protect you? Or your parents were too drunk or high to take care of you or take you to the doctor if you needed it?
q6:  Were your parents ever separated or divorced?
q7:  Was your mother or stepmother often or very often pushed, grabbed, slapped, or had something thrown at her? Or sometimes, often, or very often kicked, bitten, hit with a fist, or hit with something hard? Or ever repeatedly hit over at least a few minutes or threatened with a gun or knife?
q8:  Did you live with anyone who was a problem drinker or alcoholic, or who used street drugs?
q9:  Was a household member depressed or mentally ill, or did a household member attempt suicide?
q10: Did a household member go to prison?

Scale mapping:
0 = No
1 = Yes

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The answer for each question must be a single string value: "0" (No) or "1" (Yes).
Rules:
- For each question q1-q10, determine the score based on the user's responses.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q10 key.
- expected_output must contain exactly q1 through q10.
- conversation must be consistent with expected_output.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "q8": "0",
  "q9": "0",
  "q10": "0"
}
""",
    "hamd": """
You are a HAM-D (Hamilton Depression Rating Scale, 21-item) survey scoring assistant.
Your job is to extract answers for the HAM-D-21 based on a given conversation transcript.
The HAM-D-21 is a clinician-administered assessment that measures the severity of depression over the past week.

Each item below lists its description and its allowed score range. Different items use different scales:

q1  Depressed mood (sadness, hopelessness, helplessness, worthlessness)        scale 0-4
q2  Feelings of guilt (self-reproach, ideas of guilt, delusions of guilt)       scale 0-4
q3  Suicide (feels life not worth living, wishes dead, ideation, attempts)      scale 0-4
q4  Insomnia early (difficulty falling asleep)                                  scale 0-2
q5  Insomnia middle (waking during the night, restless sleep)                   scale 0-2
q6  Insomnia late (waking in early hours and unable to fall back asleep)        scale 0-2
q7  Work and activities (loss of interest, fatigue, decreased productivity)     scale 0-4
q8  Retardation (slowness of thought, speech, movement) - usually observed      scale 0-4
q9  Agitation (fidgeting, hand-wringing, hair-pulling) - usually observed       scale 0-4
q10 Anxiety - psychic (subjective tension, worry, irritability, fears)          scale 0-4
q11 Anxiety - somatic (GI, cardiovascular, respiratory, sweating, headaches)    scale 0-4
q12 Somatic symptoms - gastrointestinal (loss of appetite, heavy stomach)       scale 0-2
q13 Somatic symptoms - general (heaviness, fatigue, backache, muscle aches)     scale 0-2
q14 Genital symptoms (loss of libido, menstrual disturbances)                   scale 0-2
q15 Hypochondriasis (preoccupation with bodily health, conviction of illness)   scale 0-4
q16 Loss of weight (by history or measured)                                     scale 0-2
q17 Insight (recognition of being depressed/ill)                                scale 0-2
q18 Diurnal variation (whether mood is worse at a particular time of day)       scale 0-2
q19 Depersonalization and derealization (feelings of unreality, dreamlike)      scale 0-4
q20 Paranoid symptoms (suspiciousness, ideas of reference, persecution)         scale 0-4
q21 Obsessional and compulsive symptoms (intrusive thoughts, rituals)           scale 0-2

General severity guide for each scale:
0 = absent / none
1 = mild / occasional / doubtful
2 = moderate / definite
3 = severe (only for 0-4 items)
4 = very severe / incapacitating (only for 0-4 items)

Specific scoring guidance for tricky items:
- q1: 0=absent, 1=indicated only on questioning, 2=spontaneously reported verbally, 3=communicated nonverbally (facial expression, voice, tearfulness), 4=patient reports virtually only these feelings.
- q3: 0=absent, 1=feels life not worth living, 2=wishes were dead, 3=suicidal ideas or gestures, 4=attempts at suicide.
- q4/q5/q6: 0=no difficulty, 1=occasional/mild, 2=nightly/severe.
- q7: 0=no difficulty, 1=thoughts of incapacity/fatigue, 2=loss of interest in activities, 3=decreased productivity, 4=stopped working due to illness.
- q8 and q9 are clinician-observed; if not evident from the transcript, score 0.
- q15: 0=not present, 1=self-absorption (bodily), 2=preoccupation with health, 3=frequent complaints/requests for help, 4=hypochondriacal delusions.
- q16: 0=no weight loss, 1=probable weight loss associated with illness, 2=definite weight loss.
- q17: 0=acknowledges being depressed and ill, 1=acknowledges illness but attributes cause to bad food/climate/overwork/virus/need for rest, 2=denies being ill at all.
- q18: 0=no variation, 1=mild variation worse in AM or PM, 2=severe variation.
- q20: 0=none, 1=suspicious, 2=ideas of reference, 3=delusions of reference and persecution, 4=hallucinations, persecutory.

Topic order in the conversation (use this as a guide; the assistant typically asks about these in roughly this order):
mood -> crying -> guilt / self-blame -> punishment -> suicide -> sleep onset (q4) -> sleep middle (q5) -> sleep early morning (q6) -> work / activities (q7) -> anxiety / worry (q10) -> physical anxiety symptoms (q11) -> appetite (q12, infer q16) -> energy / heaviness / aches (q13) -> sex (q14) -> health concerns (q15) -> diurnal variation (q18) -> unreal / dreamlike (q19) -> paranoid (q20) -> obsessive / compulsive (q21) -> insight (q17)

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value within that item's allowed range ("0"-"4" for 0-4 items, "0"-"2" for 0-2 items).
Rules:
- Match each topic in the conversation to the correct q number based on its description above (not by response position).
- Score each item based solely on what the user said in the transcript.
- For q8 and q9, score 0 unless the user clearly describes psychomotor retardation or agitation.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q21 key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "q8": "0",
  "q9": "0",
  "q10": "0",
  "q11": "0",
  "q12": "0",
  "q13": "0",
  "q14": "0",
  "q15": "0",
  "q16": "0",
  "q17": "0",
  "q18": "0",
  "q19": "0",
  "q20": "0",
  "q21": "0"
}
""",
    "pss14": """
You are a PSS-14 survey scoring assistant.
Your job is to extract answers for the PSS-14 based on a given conversation transcript.
The Perceived Stress Scale 14-item version (PSS-14) is a clinical assessment tool that measures how often a person has felt or thought a certain way over the past month.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1   Upset because of something that happened unexpectedly
q2   Felt unable to control the important things in your life
q3   Felt nervous or stressed
q4   Dealt successfully with irritating everyday hassles            (positive item)
q5   Felt that you were effectively coping with important changes    (positive item)
q6   Felt confident about your ability to handle personal problems   (positive item)
q7   Felt that things were going your way                            (positive item)
q8   Found that you could not cope with all the things you had to do
q9   Been able to control irritations in your life                   (positive item)
q10  Felt that you were on top of things                             (positive item)
q11  Been angered by things that happened outside your control
q12  Found yourself thinking about things you still have to accomplish
q13  Been able to control the way you spend your time                (positive item)
q14  Felt that difficulties were piling up so high you could not overcome them

Scale mapping:
0 = never
1 = almost never
2 = sometimes
3 = fairly often
4 = very often

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", "3", or "4".
Rules:
- Read each user response in order. The first user response answers q1, the second answers q2, the third answers q3, and so on through q14.
- Score each response based solely on the frequency the user reported (do NOT reverse-score positive items here; record the literal frequency they expressed).
- Map natural-language frequencies to the closest scale value (e.g., "never" -> 0, "almost never" / "rarely" -> 1, "sometimes" / "occasionally" -> 2, "fairly often" / "often" -> 3, "very often" / "almost always" / "constantly" -> 4).
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q14 key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "q8": "0",
  "q9": "0",
  "q10": "0",
  "q11": "0",
  "q12": "0",
  "q13": "0",
  "q14": "0"
}
""",
    "qids": """
You are a QIDS-SR (Quick Inventory of Depressive Symptomatology - Self Report) scoring assistant.
Your job is to extract item-level scores for the QIDS-SR based on a given conversation transcript.
The QIDS-SR assesses 16 individual items covering sleep, mood, appetite, weight, concentration, self-view, death/suicide ideation, interest, energy, and psychomotor symptoms over the past seven days.

QIDS-SR item definitions and scale:
item1  (falling asleep):   0=no delay, 1=<30min less than half nights, 2=30-60min more than half nights, 3=>60min more than half nights
item2  (sleep during night): 0=no waking, 1=restless light sleep, 2=wakes 1-2x/night, 3=awake most of night
item3  (waking early):     0=no early waking, 1=<1hr early less than half nights, 2=>=1hr early more than half nights, 3=>=2hr early every night
item4  (sleeping too much): 0=no excess sleep, 1=sleeps 1hr more than usual, 2=sleeps 2hr more than usual, 3=sleeps 3+hr more than usual
item5  (sad mood):         0=no sadness, 1=sad less than half the time, 2=sad more than half the time, 3=sad nearly all the time
item6  (decreased appetite): 0=no change, 1=eats somewhat less, 2=eats much less, 3=rarely eats within 24hr
item7  (increased appetite): 0=no change, 1=eats somewhat more, 2=eats much more, 3=compelled to eat much more
item8  (decreased weight):  0=no change, 1=1-2lb decrease, 2=3-4lb decrease, 3=>5lb decrease
item9  (increased weight):  0=no change, 1=1-2lb increase, 2=3-4lb increase, 3=>5lb increase
item10 (concentration):    0=no change, 1=occasional difficulty, 2=difficulty most of the time, 3=unable to concentrate
item11 (view of self):     0=no change, 1=self-critical more than usual, 2=believes they cause problems, 3=constantly believes they are a failure
item12 (death/suicide):    0=no thoughts, 1=feels life is empty, 2=thinks of death/suicide several times/week, 3=thinks of death/suicide daily
item13 (interest):         0=no change, 1=less interested in some activities, 2=interested in only 1-2 activities, 3=no interest in any activity
item14 (energy):           0=no change, 1=tires more easily, 2=effort needed for routine activities, 3=cannot carry out routine activities
item15 (slowed down):      0=no slowing, 1=slightly slow, 2=clearly slow, 3=cannot move without assistance
item16 (restless):         0=no restlessness, 1=fidgety, 2=wringing hands/can't sit still, 3=moving constantly, can't stay seated

Important scoring rules:
- For sleep domain: only one of items 1-4 will be non-zero (use the dominant sleep disturbance)
- For appetite/weight domain: only items 6-7 OR items 8-9 will be non-zero (not both directions)
- For psychomotor domain: only one of items 15-16 will be non-zero (slowing OR restlessness)

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The score for each item must be a single string value: "0", "1", "2", or "3".
Rules:
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any item1-item16 key.
Return JSON in exactly this shape:
{
  "item1": "0",
  "item2": "0",
  "item3": "0",
  "item4": "0",
  "item5": "0",
  "item6": "0",
  "item7": "0",
  "item8": "0",
  "item9": "0",
  "item10": "0",
  "item11": "0",
  "item12": "0",
  "item13": "0",
  "item14": "0",
  "item15": "0",
  "item16": "0"
}
"""
}

EOS_TOKEN = tokenizer.eos_token

def format_dataset(conversations, scores, task_labels):
    texts = []
    for conv, score, task in zip(conversations, scores, task_labels):
        prompt = prompts[task].strip()
        conv_str = json.dumps(conv, ensure_ascii=False)
        score_str = json.dumps(score, ensure_ascii=False)
        text = prompt + f"\n\nInput:\n{conv_str}\n\nOutput:\n{score_str}" + EOS_TOKEN
        texts.append(text)
    return texts

# Format the training split
formatted_texts = format_dataset(conv_train, score_train, task_train)

# Convert to a Hugging Face Dataset object which SFTTrainer requires
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"Successfully created HuggingFace Dataset with {len(dataset)} examples.")

### 4. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 3,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.05,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 5. Export to Ollama (GGUF)

In [ ]:
# Save to 8bit Q8_0
model.save_pretrained_gguf("model_for_ollama", tokenizer, quantization_method = "q8_0")
# Experimental AWQ for vllm
# model.save_pretrained_merged("model_awq", tokenizer, save_method = "awq")

print("Done! You can now download the .gguf file from the 'model_for_ollama' folder and run it in Ollama using a Modelfile.")